# Serving Model GGUF dengan llama.cpp dan Cache Jawaban di SQL

Notebook ini menjalankan model GGUF hasil fine-tuning (Llama 3.1 8B, Q4_K_M) dengan **llama.cpp**, lalu menambahkan **cache jawaban di database SQL** di depannya.

**Yang dikerjakan dan diukur**
1. Memuat GGUF (hasil ekspor notebook 01 di Google Drive; jika tidak ada, memakai model fallback dari Hugging Face).
2. Membandingkan kecepatan generasi **CPU vs GPU** pada model dan prompt yang sama.
3. Membangun cache jawaban dengan kunci berbasis hash yang memuat instruksi, input, model, dan parameter decoding. Tersedia backend **SQLite** (default, tanpa setup) dan **MySQL** (opsional).
4. Mengukur latensi **cache miss vs cache hit**.
5. Menguji risiko cache: jawaban salah yang tersimpan akan disajikan terus. Pengujian memakai soal kecil dengan kunci jawaban yang bisa dicek otomatis.

**Catatan keamanan:** kredensial database tidak boleh ditulis di dalam notebook. Untuk MySQL, isi Colab Secrets (`MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, `MYSQL_DB`); jika belum ada, notebook meminta input manual.

> Runtime: **T4 GPU**. Instalasi llama-cpp-python dengan CUDA dikompilasi dari source dan bisa memakan waktu belasan menit atau lebih. Selama kompilasi sel terlihat diam; itu normal.

## 1. Instalasi

`llama-cpp-python` dikompilasi dengan dukungan CUDA untuk arsitektur GPU yang terdeteksi (misalnya `75` untuk T4). Tanpa langkah ini, pustaka berjalan di CPU meskipun runtime memakai GPU.

In [1]:
import os, gc, json, time, glob, hashlib, sqlite3
import torch

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    major, minor = torch.cuda.get_device_capability(0)
    os.environ["CMAKE_ARGS"] = f"-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES={major}{minor}"
    print(f"GPU: {torch.cuda.get_device_name(0)} (compute capability {major}.{minor})")
else:
    os.environ["CMAKE_ARGS"] = ""
    print("GPU tidak terdeteksi. llama-cpp-python akan dibangun untuk CPU.")
print("CMAKE_ARGS =", os.environ["CMAKE_ARGS"] or "(kosong)")

GPU: NVIDIA A100-SXM4-40GB (compute capability 8.0)
CMAKE_ARGS = -DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=80


In [2]:
!pip install -q llama-cpp-python huggingface_hub mysql-connector-python --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 164.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 233.0 MB/s eta 0:00:00


## 2. Lokasi model

Urutan pencarian: file `.gguf` di folder Google Drive hasil notebook 01. Jika tidak ada, model fallback diunduh dari Hugging Face.
Model fallback **bukan** hasil training di notebook ini; sumbernya dicetak agar jelas mana yang dipakai.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/AI-Engineer/LLM/llama31-8b-alpaca-id"
FALLBACK_REPO, FALLBACK_FILE = "rubythalib33/llama3_1_8b_finetuned_bahasa_indonesia", "unsloth.Q4_K_M.gguf"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

local = sorted(glob.glob(f"{DRIVE_DIR}/**/*.gguf", recursive=True),
               key=lambda p: ("q4_k_m" not in p.lower(), p))
if local:
    MODEL_PATH, MODEL_SOURCE = local[0], "drive (hasil ekspor sendiri)"
else:
    from huggingface_hub import hf_hub_download
    MODEL_PATH = hf_hub_download(repo_id=FALLBACK_REPO, filename=FALLBACK_FILE)
    MODEL_SOURCE = f"fallback Hugging Face: {FALLBACK_REPO}"

MODEL_TAG = os.path.basename(MODEL_PATH)
print("Sumber model :", MODEL_SOURCE)
print("File         :", MODEL_PATH)
print(f"Ukuran       : {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB")

Mounted at /content/drive
Sumber model : drive (hasil ekspor sendiri)
File         : /content/drive/MyDrive/AI-Engineer/LLM/llama31-8b-alpaca-id/meta-llama-3.1-8b.Q4_K_M.gguf
Ukuran       : 4.92 GB


## 3. Format prompt dan parameter decoding

Model dilatih dengan prompt **Alpaca mentah** (tanpa chat template), jadi inferensi memakai `create_completion` dengan template yang sama. Memakai `create_chat_completion` pada model tanpa chat template bawaan membuat llama.cpp memilih format cadangan, yang berbeda dari format training.

Decoding **greedy** (`temperature=0`) dipakai supaya jawaban deterministik. Ini juga yang membuat cache bermakna: pertanyaan yang sama dan model yang sama akan menghasilkan jawaban yang sama.

In [4]:
ALPACA = '''Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan masukan yang memberikan konteks lebih lanjut. Tulis tanggapan yang melengkapi permintaan dengan tepat.

### Instruction:
{}

### Input:
{}

### Response:
{}'''
STOP = ["### Instruction:"]

BENCH_PROMPT = ALPACA.format("Apa itu algoritma pemrograman?", "", "")

## 4. Benchmark CPU vs GPU

Model yang sama dimuat dua kali: `n_gpu_layers=0` (semua layer di CPU) dan `n_gpu_layers=-1` (semua layer di GPU). Prompt dan jumlah token maksimum sama.
Waktu yang diukur mencakup evaluasi prompt dan generasi, dibagi jumlah token yang dihasilkan. Pengukuran satu kali; hasilnya bisa bervariasi antar sesi Colab.

In [5]:
from llama_cpp import Llama
try:
    from llama_cpp import llama_supports_gpu_offload
    GPU_OFFLOAD = HAS_GPU and bool(llama_supports_gpu_offload())
except Exception:
    GPU_OFFLOAD = HAS_GPU
print("GPU offload didukung build ini:", GPU_OFFLOAD)
if HAS_GPU and not GPU_OFFLOAD:
    print("Peringatan: GPU ada, tetapi llama-cpp-python terpasang tanpa CUDA. Ulangi instalasi (bagian 1) agar GPU terpakai.")

def benchmark(n_gpu_layers, max_tokens=64):
    llm = Llama(model_path=MODEL_PATH, n_ctx=2048, n_gpu_layers=n_gpu_layers,
                n_threads=os.cpu_count(), verbose=False)
    t0 = time.perf_counter()
    out = llm.create_completion(BENCH_PROMPT, max_tokens=max_tokens, temperature=0.0, stop=STOP)
    dt = time.perf_counter() - t0
    n = out["usage"]["completion_tokens"]
    del llm; gc.collect()
    return dict(tokens=n, seconds=round(dt, 2), tok_per_s=round(n / dt, 2))

bench = {}
print("CPU (n_gpu_layers=0)  ...")
bench["cpu"] = benchmark(0)
print("   ", bench["cpu"])
if GPU_OFFLOAD:
    print("GPU (n_gpu_layers=-1) ...")
    bench["gpu"] = benchmark(-1)
    print("   ", bench["gpu"])
    bench["speedup_x"] = round(bench["gpu"]["tok_per_s"] / bench["cpu"]["tok_per_s"], 1)
    print(f"Speedup GPU vs CPU: {bench['speedup_x']}x")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 40441 MiB):
  Device 0: NVIDIA A100-SXM4-40GB, compute capability 8.0, VMM: yes, VRAM: 40441 MiB


GPU offload didukung build ini: True
CPU (n_gpu_layers=0)  ...
    {'tokens': 64, 'seconds': 11.62, 'tok_per_s': 5.51}
GPU (n_gpu_layers=-1) ...
    {'tokens': 64, 'seconds': 0.55, 'tok_per_s': 115.47}
Speedup GPU vs CPU: 21.0x


## 5. Memuat model untuk serving

Konteks (`n_ctx`) diset eksplisit. Nilai bawaan pustaka sangat kecil dibanding kemampuan model, sehingga prompt panjang bisa terpotong atau gagal.

In [6]:
llm = Llama(model_path=MODEL_PATH, n_ctx=4096,
            n_gpu_layers=-1 if GPU_OFFLOAD else 0,
            n_threads=os.cpu_count(), verbose=False)
print("n_ctx =", llm.n_ctx(), "| GPU offload =", GPU_OFFLOAD)

n_ctx = 4096 | GPU offload = True


## 6. Lapisan cache

**Kunci cache** adalah hash SHA-256 dari: instruksi + input (dinormalisasi: spasi dirapikan dan huruf dibuat kecil), **tag model**, dan **parameter decoding**.
Dengan begitu, mengganti model atau parameter otomatis membuat kunci baru dan tidak menyajikan jawaban lama.

Batasan yang perlu dipahami: cache ini **exact match**. Kalimat berbeda dengan makna sama tetap dianggap pertanyaan baru. Untuk itu diperlukan semantic cache (embedding), yang di luar cakupan notebook ini.

Dua pengaman:
- Hanya jawaban yang selesai wajar (`finish_reason == "stop"`) yang disimpan. Jawaban yang terpotong karena batas token tidak di-cache.
- `ttl_days` opsional membuat entri lama dianggap kedaluwarsa.

In [7]:
def normalize(s):
    return " ".join((s or "").split()).casefold()

def make_key(instruction, input_data, model_tag, params):
    payload = json.dumps({"i": normalize(instruction), "x": normalize(input_data),
                          "m": model_tag, "p": params}, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

class SQLiteCache:
    name = "sqlite"
    def __init__(self, path="cache.db"):
        self.conn = sqlite3.connect(path, check_same_thread=False)
        self.conn.execute('''CREATE TABLE IF NOT EXISTS cached_chat (
            cache_key TEXT PRIMARY KEY, instruction TEXT NOT NULL, input_data TEXT NOT NULL,
            response TEXT NOT NULL, model_tag TEXT NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP)''')
        self.conn.commit()
    def get(self, key, ttl_days=None):
        sql, args = "SELECT response FROM cached_chat WHERE cache_key = ?", [key]
        if ttl_days is not None:
            sql += " AND created_at >= datetime('now', ?)"
            args.append(f"-{int(ttl_days)} days")
        row = self.conn.execute(sql, args).fetchone()
        return row[0] if row else None
    def put(self, key, instruction, input_data, response, model_tag):
        self.conn.execute("REPLACE INTO cached_chat (cache_key, instruction, input_data, response, model_tag) VALUES (?,?,?,?,?)",
                          (key, instruction, input_data, response, model_tag))
        self.conn.commit()
    def delete(self, key):
        self.conn.execute("DELETE FROM cached_chat WHERE cache_key = ?", (key,)); self.conn.commit()
    def count(self):
        return self.conn.execute("SELECT COUNT(*) FROM cached_chat").fetchone()[0]
    def clear(self):
        self.conn.execute("DELETE FROM cached_chat"); self.conn.commit()

class MySQLCache:
    name = "mysql"
    def __init__(self, host, port, user, password, database):
        import mysql.connector
        self.cfg = dict(host=host, port=int(port or 3306), user=user, password=password,
                        database=database, autocommit=True)
        self.conn = mysql.connector.connect(**self.cfg)
        self._exec('''CREATE TABLE IF NOT EXISTS cached_chat (
            cache_key CHAR(64) NOT NULL PRIMARY KEY, instruction TEXT NOT NULL, input_data TEXT NOT NULL,
            response MEDIUMTEXT NOT NULL, model_tag VARCHAR(255) NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP) DEFAULT CHARSET=utf8mb4''')
    def _exec(self, sql, args=None, fetch=False):
        self.conn.ping(reconnect=True, attempts=3, delay=1)
        cur = self.conn.cursor()
        cur.execute(sql, args or ())
        row = cur.fetchone() if fetch else None
        cur.close()
        return row
    def get(self, key, ttl_days=None):
        sql, args = "SELECT response FROM cached_chat WHERE cache_key = %s", [key]
        if ttl_days is not None:
            sql += " AND created_at >= (NOW() - INTERVAL %s DAY)"
            args.append(int(ttl_days))
        row = self._exec(sql, args, fetch=True)
        return row[0] if row else None
    def put(self, key, instruction, input_data, response, model_tag):
        self._exec("REPLACE INTO cached_chat (cache_key, instruction, input_data, response, model_tag) VALUES (%s,%s,%s,%s,%s)",
                   (key, instruction, input_data, response, model_tag))
    def delete(self, key):
        self._exec("DELETE FROM cached_chat WHERE cache_key = %s", (key,))
    def count(self):
        return self._exec("SELECT COUNT(*) FROM cached_chat", fetch=True)[0]
    def clear(self):
        self._exec("DELETE FROM cached_chat")

In [8]:
USE_MYSQL = False   # ubah ke True untuk memakai MySQL (kredensial dari Colab Secrets, atau input manual)

def load_mysql_config():
    from getpass import getpass
    keys = ["MYSQL_HOST", "MYSQL_PORT", "MYSQL_USER", "MYSQL_PASSWORD", "MYSQL_DB"]
    cfg = {}
    try:
        from google.colab import userdata
        for k in keys:
            try:
                cfg[k] = userdata.get(k)
            except Exception:
                cfg[k] = None
    except Exception:
        cfg = {k: None for k in keys}
    for k in keys:
        if not cfg.get(k):
            ask = getpass if "PASSWORD" in k else input
            cfg[k] = ask(f"{k}: ").strip()
    return cfg

if USE_MYSQL:
    c = load_mysql_config()
    cache = MySQLCache(c["MYSQL_HOST"], c["MYSQL_PORT"], c["MYSQL_USER"], c["MYSQL_PASSWORD"], c["MYSQL_DB"])
else:
    cache = SQLiteCache("/content/cache.db")
print("Backend cache:", cache.name, "| entri saat ini:", cache.count())

Backend cache: sqlite | entri saat ini: 0


## 7. Fungsi `chat_complete`

In [9]:
def cache_key_for(instruction, input_data="", max_tokens=256):
    return make_key(instruction, input_data, MODEL_TAG, {"temperature": 0.0, "max_tokens": max_tokens})

def chat_complete(instruction, input_data="", use_cache=True, max_tokens=256, ttl_days=None):
    key = cache_key_for(instruction, input_data, max_tokens)
    t0 = time.perf_counter()
    if use_cache:
        hit = cache.get(key, ttl_days)
        if hit is not None:
            return dict(text=hit, source="cache", seconds=time.perf_counter() - t0, tokens=None, key=key)
    out = llm.create_completion(ALPACA.format(instruction, input_data, ""),
                                max_tokens=max_tokens, temperature=0.0, stop=STOP)
    choice = out["choices"][0]
    text = choice["text"].strip()
    finished = choice.get("finish_reason") == "stop"
    if use_cache and finished and text:
        cache.put(key, instruction, input_data, text, MODEL_TAG)
    return dict(text=text, source="model", seconds=time.perf_counter() - t0,
                tokens=out["usage"]["completion_tokens"], finish_reason=choice.get("finish_reason"), key=key)

## 8. Latensi: cache miss vs cache hit

Setiap prompt dijalankan dua kali. Entri lama dihapus lebih dulu agar panggilan pertama pasti miss.

In [10]:
import pandas as pd

DEMO = [("siapa presiden pertama di indonesia?", ""),
        ("ibu kota indonesia adalah", ""),
        ("Apa itu algoritma pemrograman?", "")]

rows = []
for ins, inp in DEMO:
    cache.delete(cache_key_for(ins, inp))
    first, second = chat_complete(ins, inp), chat_complete(ins, inp)
    rows.append(dict(prompt=ins, tokens=first["tokens"], miss_s=round(first["seconds"], 2),
                     hit_ms=round(second["seconds"] * 1000, 2), sumber_kedua=second["source"],
                     jawaban_sama=(first["text"] == second["text"]),
                     speedup_x=round(first["seconds"] / max(second["seconds"], 1e-9))))
latency_df = pd.DataFrame(rows)
print(latency_df.to_string(index=False))
print("\nEntri di cache:", cache.count())

                              prompt  tokens  miss_s  hit_ms sumber_kedua  jawaban_sama  speedup_x
siapa presiden pertama di indonesia?      25    0.25    0.06        cache          True       4124
           ibu kota indonesia adalah       7    0.08    0.06        cache          True       1475
      Apa itu algoritma pemrograman?     192    1.45    0.06        cache          True      25520

Entri di cache: 3


## 9. Risiko cache: jawaban salah ikut tersimpan

Cache tidak tahu benar atau salahnya sebuah jawaban. Untuk menguji ini dipakai soal kecil dengan kunci jawaban yang bisa dicek otomatis: daftar lima nama diberikan lewat kolom `input`, lalu model ditanya urutan ke-1 sampai ke-5.
Pengecekan lewat kode: jawaban dianggap benar jika memuat nama yang diharapkan dan tidak memuat nama lain dari daftar.
Bagian ini dijalankan **tanpa cache** agar yang diukur murni kemampuan model.

In [11]:
NAMES = ["hashirama", "uchiha", "choji", "minato", "naruto"]
LIST_TXT = "urutan presiden konoha adalah " + ", ".join(NAMES)
ordinal_q = lambda n: f"siapa presiden ke-{n} di konoha"

rows = []
for n in range(1, len(NAMES) + 1):
    r = chat_complete(ordinal_q(n), LIST_TXT, use_cache=False, max_tokens=64)
    mentioned = [x for x in NAMES if x in r["text"].lower()]
    rows.append(dict(urutan=n, diharapkan=NAMES[n - 1], jawaban=r["text"], benar=(mentioned == [NAMES[n - 1]])))
konoha_df = pd.DataFrame(rows)
print(konoha_df.to_string(index=False))
konoha_acc = round(100 * konoha_df["benar"].mean(), 1)
print(f"\nAkurasi: {konoha_acc}% ({int(konoha_df['benar'].sum())}/{len(konoha_df)})")

 urutan diharapkan                                            jawaban  benar
      1  hashirama       Presiden Konoha ke-1 adalah Hashirama Senju.   True
      2     uchiha             Presiden ke-2 di Konoha adalah Uchiha.   True
      3      choji             Presiden ke-3 di Konoha adalah Uchiha.  False
      4     minato Presiden keempat di Konoha adalah Minato Namikaze.   True
      5     naruto    Presiden ke-5 di Konoha adalah Minato Namikaze.  False

Akurasi: 60.0% (3/5)


In [12]:
wrong = konoha_df[~konoha_df["benar"]]
if wrong.empty:
    print("Semua jawaban benar pada uji ini, jadi tidak ada contoh jawaban salah untuk didemonstrasikan.")
else:
    n = int(wrong.iloc[0]["urutan"])
    q = ordinal_q(n)
    key = cache_key_for(q, LIST_TXT, 64)
    cache.delete(key)

    first  = chat_complete(q, LIST_TXT, max_tokens=64)
    second = chat_complete(q, LIST_TXT, max_tokens=64)
    print(f"Pertanyaan      : {q}")
    print(f"Diharapkan      : {NAMES[n - 1]}")
    print(f"Panggilan 1 ({first['source']:>5}): {first['text']}")
    print(f"Panggilan 2 ({second['source']:>5}): {second['text']}")

    cache.delete(key)
    third = chat_complete(q, LIST_TXT, max_tokens=64)
    print(f"Setelah delete  ({third['source']:>5}): {third['text']}")
    print("\nMenghapus entri hanya membuang salinan yang tersimpan. Karena decoding greedy, model yang sama "
          "menghasilkan jawaban yang sama lagi. Yang memperbaiki hasil adalah mengubah prompt, data, atau model.")

Pertanyaan      : siapa presiden ke-3 di konoha
Diharapkan      : choji
Panggilan 1 (model): Presiden ke-3 di Konoha adalah Uchiha.
Panggilan 2 (cache): Presiden ke-3 di Konoha adalah Uchiha.
Setelah delete  (model): Presiden ke-3 di Konoha adalah Uchiha.

Menghapus entri hanya membuang salinan yang tersimpan. Karena decoding greedy, model yang sama menghasilkan jawaban yang sama lagi. Yang memperbaiki hasil adalah mengubah prompt, data, atau model.


## 10. Ringkasan hasil

In [13]:
summary = dict(
    model_source   = MODEL_SOURCE,
    model_file     = MODEL_TAG,
    model_size_gb  = round(os.path.getsize(MODEL_PATH) / 1e9, 2),
    n_ctx          = 4096,
    cache_backend  = cache.name,
    gpu_offload    = GPU_OFFLOAD,
    benchmark      = bench,
    cache_latency  = latency_df.to_dict(orient="records"),
    konoha_accuracy_pct = konoha_acc,
)
os.makedirs(DRIVE_DIR, exist_ok=True)
with open(f"{DRIVE_DIR}/serving_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print(json.dumps(summary, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)))

{
  "model_source": "drive (hasil ekspor sendiri)",
  "model_file": "meta-llama-3.1-8b.Q4_K_M.gguf",
  "model_size_gb": 4.92,
  "n_ctx": 4096,
  "cache_backend": "sqlite",
  "gpu_offload": true,
  "benchmark": {
    "cpu": {
      "tokens": 64,
      "seconds": 11.62,
      "tok_per_s": 5.51
    },
    "gpu": {
      "tokens": 64,
      "seconds": 0.55,
      "tok_per_s": 115.47
    },
    "speedup_x": 21.0
  },
  "cache_latency": [
    {
      "prompt": "siapa presiden pertama di indonesia?",
      "tokens": 25,
      "miss_s": 0.25,
      "hit_ms": 0.06,
      "sumber_kedua": "cache",
      "jawaban_sama": true,
      "speedup_x": 4124
    },
    {
      "prompt": "ibu kota indonesia adalah",
      "tokens": 7,
      "miss_s": 0.08,
      "hit_ms": 0.06,
      "sumber_kedua": "cache",
      "jawaban_sama": true,
      "speedup_x": 1475
    },
    {
      "prompt": "Apa itu algoritma pemrograman?",
      "tokens": 192,
      "miss_s": 1.45,
      "hit_ms": 0.06,
      "sumber_kedua": 

## 11. Catatan dan keterbatasan

- **Exact match.** Cache tidak mengenali kalimat berbeda dengan makna sama. Semantic cache membutuhkan embedding dan ambang kemiripan.
- **Cache tidak menilai kebenaran.** Jawaban salah yang selesai wajar tetap tersimpan. Untuk produksi, tambahkan validasi, umpan balik pengguna, atau TTL pendek.
- **Deterministik.** Cache hanya masuk akal untuk decoding greedy. Dengan sampling, jawaban yang sama bisa berbeda tiap panggilan, dan cache akan membekukan satu variasi.
- **Benchmark satu kali.** Kecepatan CPU dan GPU bervariasi antar sesi Colab; ulangi beberapa kali sebelum menyimpulkan.
- **Privasi.** Cache menyimpan teks pertanyaan pengguna. Pertimbangkan kebijakan retensi dan anonimisasi sebelum dipakai pada data nyata.
- **Concurrency.** Backend di sini sederhana (satu koneksi). Lalu lintas paralel butuh connection pool dan penanganan race condition.

---

*Materi berdasarkan kurikulum Machine Learning on Production, rubythalib.ai.*